# Fine-Tune a Vision Transformer on Cassava Leaves

**Course:** Session 15 of [Master Transformers from Scratch: BERT, ViT & AI Search](https://www.udemy.com/course/master-transformers-from-scratch-bert-vit-ai-search/)  
**Task:** 5-class image classification (cassava leaf disease)  
**Model:** `google/vit-base-patch16-224`  
**Platform:** Kaggle Notebook + GPU  

### Setup on Kaggle
1. Open [Cassava Leaf Disease Classification](https://www.kaggle.com/c/cassava-leaf-disease-classification) and accept the rules
2. Create a notebook from the competition **Code** tab (or upload this file)
3. **Session options → Accelerator → GPU T4**
4. **Session options → Internet → On** (needed to download ViT weights)
5. Confirm Input contains `train.csv`, `train_images/`, and `test_images/`
6. Run All

### Classroom note
The full train set has about 21k images. This notebook uses a **stratified sample** so a free T4 can finish in a classroom window. Set `MAX_TRAIN = None` to use every labeled image.


## Cell 1 — Setup and imports

We load PyTorch, pandas, scikit-learn, Hugging Face `datasets`, and the vision helpers.  
`AutoImageProcessor` replaces the text tokenizer.  
`AutoModelForImageClassification` loads a pretrained ViT and attaches a new 5-class head.

In [ ]:
import os
import glob
import json
import inspect
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)

SEED = 42
MODEL_NAME = "google/vit-base-patch16-224"
NUM_LABELS = 5
# Classroom speed: use 4000 labeled images. Set to None for the full train set.
MAX_TRAIN = 4000
NUM_EPOCHS = 2
BATCH_SIZE = 16

print("MODEL_NAME =", MODEL_NAME)
print("MAX_TRAIN =", MAX_TRAIN)


## Cell 2 — Find the competition folder and load labels

On Kaggle the mount path can vary. We search for `train.csv` under `/kaggle/input` instead of hard-coding the folder name.  
We also load the official label map so class numbers become disease names.

In [ ]:
print("contents of /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "not on Kaggle")

candidates = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + ["./train.csv"]
DATA = next(os.path.dirname(c) or "." for c in candidates if os.path.exists(c))
print("using DATA =", DATA)

train_df = pd.read_csv(os.path.join(DATA, "train.csv"))
sample_sub = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))

label_map_path = os.path.join(DATA, "label_num_to_disease_map.json")
with open(label_map_path) as f:
    id2label = {int(k): v for k, v in json.load(f).items()}
label2id = {v: k for k, v in id2label.items()}

print("train:", train_df.shape)
print("label map:")
for i in range(NUM_LABELS):
    print(f"  {i}: {id2label[i]}")
train_df.head(3)


## Cell 3 — Explore class balance and show one leaf image

Cassava Mosaic Disease dominates the labels. Stratified splits keep that balance in train and validation.  
We also open one JPEG so you can see the raw input the Vision Transformer will receive.

In [ ]:
print("label counts:")
print(train_df["label"].value_counts().sort_index())
print("\nfriendly counts:")
print(train_df["label"].map(id2label).value_counts())

demo_row = train_df.iloc[0]
demo_path = os.path.join(DATA, "train_images", demo_row["image_id"])
demo_img = Image.open(demo_path).convert("RGB")
print(f"\nexample file: {demo_row['image_id']}")
print(f"label: {demo_row['label']} → {id2label[int(demo_row['label'])]}")
print(f"image size: {demo_img.size}")
demo_img


## Cell 4 — Stratified split (and optional classroom sample)

We keep 20% of the chosen rows for validation.  
If `MAX_TRAIN` is set, we first draw a stratified sample so every disease still appears.

In [ ]:
work_df = train_df
if MAX_TRAIN is not None and MAX_TRAIN < len(train_df):
    work_df, _ = train_test_split(
        train_df,
        train_size=MAX_TRAIN,
        random_state=SEED,
        stratify=train_df["label"],
    )
    print(f"sampled {len(work_df)} of {len(train_df)} labeled images for the classroom run")

tr_df, va_df = train_test_split(
    work_df,
    test_size=0.2,
    random_state=SEED,
    stratify=work_df["label"],
)
print(f"train={len(tr_df)}  validation={len(va_df)}")
print("validation label counts:")
print(va_df["label"].value_counts().sort_index())


## Cell 5 — Load the image processor

The processor is the vision twin of a tokenizer.  
It resizes the photo to 224 by 224, converts pixels to a tensor, and normalizes with the ImageNet statistics the pretrained ViT expects.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
print(processor)

demo_inputs = processor(images=demo_img, return_tensors="pt")
print("pixel_values shape:", tuple(demo_inputs["pixel_values"].shape))
# shape is (batch=1, channels=3, height=224, width=224)


## Cell 6 — Build Hugging Face image datasets

Each example stores a file path and an integer label.  
The transform opens the JPEG, runs the processor, and returns `pixel_values` plus `labels` for the Trainer.

In [ ]:
def make_path_dataset(df):
    paths = [os.path.join(DATA, "train_images", name) for name in df["image_id"].tolist()]
    return Dataset.from_dict({"image_path": paths, "labels": df["label"].tolist()})


def transform_batch(batch):
    images = [Image.open(p).convert("RGB") for p in batch["image_path"]]
    out = processor(images=images, return_tensors="pt")
    batch["pixel_values"] = out["pixel_values"]
    return batch


train_ds = make_path_dataset(tr_df)
val_ds = make_path_dataset(va_df)
train_ds.set_transform(transform_batch)
val_ds.set_transform(transform_batch)

print(train_ds)
print("one training example keys:", train_ds[0].keys())
print("pixel_values:", tuple(train_ds[0]["pixel_values"].shape), "label:", train_ds[0]["labels"])


## Cell 7 — Load pretrained ViT for 5-class classification

`google/vit-base-patch16-224` already knows general ImageNet features.  
We replace the final classifier so it outputs five cassava logits.  
`ignore_mismatched_sizes=True` lets Hugging Face drop the old 1000-class head cleanly.

In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={str(k): v for k, v in id2label.items()},
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print(model.classifier)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable:,}")


## Cell 8 — Metrics, TrainingArguments, and Trainer

Accuracy and macro F1 both matter because the classes are imbalanced.  
We keep checkpoints small and turn off external logging.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


eval_key = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
    else "evaluation_strategy"
)

args = TrainingArguments(
    output_dir="cassava-vit",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    fp16=True,
    **{eval_key: "epoch"},
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DefaultDataCollator(),
    compute_metrics=compute_metrics,
)
print("Trainer ready.")


## Cell 9 — Fine-tune and evaluate

This is the expensive cell. On a T4 with `MAX_TRAIN=4000` and two epochs it usually finishes in a classroom-friendly window.  
Watch the validation accuracy and macro F1 after each epoch.

In [ ]:
train_result = trainer.train()
print(train_result)
metrics = trainer.evaluate()
print(metrics)


## Cell 10 — Predict on test images and write submission.csv

The competition test folder may contain only a tiny public stub in the notebook environment.  
We still write `submission.csv` in the required format.  
If late submission is open on the competition page, upload this file there.

In [ ]:
test_dir = os.path.join(DATA, "test_images")
test_files = sorted(
    f for f in os.listdir(test_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))
) if os.path.isdir(test_dir) else []
print("test images found:", len(test_files))

if not test_files:
    # Fall back to the sample submission ids if the folder is empty in this mount.
    test_files = sample_sub["image_id"].tolist()
    print("using image ids from sample_submission.csv")

preds = []
model.eval()
import torch

device = next(model.parameters()).device
for name in test_files:
    path = os.path.join(test_dir, name)
    if not os.path.exists(path):
        preds.append(3)  # majority-class fallback if a listed file is missing
        continue
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    preds.append(int(logits.argmax(dim=-1).item()))

submission = pd.DataFrame({"image_id": test_files, "label": preds})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv")


## Done

You completed the Session 15 loop:

1. patches + CLS + the same transformer encoder idea from the NLP sessions
2. pretrained Vision Transformer weights from Hugging Face
3. a new 5-class head for cassava diseases
4. fine-tuning on leaf photos inside a Kaggle GPU notebook
5. a `submission.csv` in the competition format

**Optional next steps:** set `MAX_TRAIN = None`, add Albumentations, or ensemble ViT with a CNN the way many Cassava medal solutions did.
